Tools

Models can request to call that the perform takes such as fetching from a databse, searching the web , or running code.Tools are paring of:                                                                                1. A schema , including the name of the tool, a description, and/or argument definations (oftebn a JSON schema)                                                                            2. A function or coroutine to execute 

In [5]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv('GROQ_API_KEY')

model = init_chat_model("groq:openai/gpt-oss-120b")

In [7]:
from langchain.tools import tool 

@tool
def get_weather(location:str)->str:
    """get the weather at a location"""
    return f"It's sunny un {location}"

model_with_tools = model.bind_tools([get_weather])    

In [9]:

resposne = model_with_tools.invoke("What's the weather like in Boston?")
print(resposne)
for tool_cell in resposne.tool_calls:
    print(f"Tool {tool_cell['name']}")
    print(f"Args {tool_cell['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to get weather for Boston using the function get_weather. Then respond.', 'tool_calls': [{'id': 'fc_7895a23d-96fe-4a70-9a05-5998d17e8e70', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 127, 'total_tokens': 171, 'completion_time': 0.095247308, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.005502229, 'prompt_tokens_details': None, 'queue_time': 0.311514419, 'total_time': 0.100749537}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_df9620fe21', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a15506-565f-7051-a5af-aeff2e7e0068-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_7895a23d-96fe-4a70-9a05-5998d17e8e70', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 1

### tools execution Loops

In [11]:
#step 1 :Model generate tool call
message = [{"role":"user" , "content":"what's a weather in Boston?"}]
ai_msg = model_with_tools.invoke(message)
message.append(ai_msg)

#step 2 : Execute tool and collect result
for tool_call  in ai_msg.tool_calls:
    #Exceute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    message.append(tool_result)
#Step 3:Pass result back to model for final resposne
final_response = model_with_tools.invoke(message)
print(final_response.text)    

The current weather in Boston is sunny. Enjoy the nice day!


In [15]:
message

[{'role': 'user', 'content': "what's a weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "what\'s a weather in Boston?" Probably wants current weather. Use function get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_3da39a58-1900-42f8-b5ea-12fc7c3bb74c', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 127, 'total_tokens': 180, 'completion_time': 0.112669764, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.005615339, 'prompt_tokens_details': None, 'queue_time': 0.349042646, 'total_time': 0.118285103}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_02b0d31eca', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a15516-89c1-70c3-86a1-0881060e10a0-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '